In [ ]:
from datasets import load_dataset
import pandas as pd
import textwrap
import os

DATASET_NAME = "ibrahimhamamci/CT-RATE"
OUTPUT_DIR = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load datasets
print("Loading CT-RATE dataset...")

metadata_dataset = load_dataset(DATASET_NAME, "metadata")
reports_dataset = load_dataset(DATASET_NAME, "reports")
labels_dataset = load_dataset(DATASET_NAME, "labels")

print(f"✓ Reports: {len(reports_dataset['train'])} entries")
print(f"✓ Metadata: {len(metadata_dataset['train'])} entries")
print(f"✓ Labels: {len(labels_dataset['train'])} entries")

In [ ]:
# Convert to DataFrames
df_reports = pd.DataFrame(reports_dataset['train'])
df_metadata = pd.DataFrame(metadata_dataset['train'])
df_labels = pd.DataFrame(labels_dataset['train'])

print(f"DataFrames created:")
print(f"  Reports: {df_reports.shape}")
print(f"  Metadata: {df_metadata.shape}")
print(f"  Labels: {df_labels.shape}")

In [ ]:
# Merge all data
df_combined = df_reports.merge(df_metadata, on='VolumeName', how='inner')
df_combined = df_combined.merge(df_labels, on='VolumeName', how='inner')

print(f"✓ Combined dataset: {df_combined.shape}")
print(f"  {len(df_combined)} rows, {len(df_combined.columns)} columns")

In [ ]:
# Analyze single report
sample_idx = 7 # report 7 has clinical information
sample = df_combined.iloc[sample_idx]

print(f"Sample Report Analysis (index {sample_idx}):")
print(f"VolumeName: {sample['VolumeName']}\n")

for field in ['Findings_EN', 'Impressions_EN', 'ClinicalInformation_EN', 'Technique_EN']:
    text = str(sample[field])
    print(f"{field}: ({len(text.split())} words, {len(text)} chars)")
    print(textwrap.fill(text, width=80)[:200] + "...\n")

In [ ]:
# Structural checks
print("Structural Analysis:")
print(f"  Duplicates: {df_combined['VolumeName'].duplicated().sum()}")
print(f"  Missing values:")

nan_counts = df_combined.isnull().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)

if nan_counts.empty:
    print("    None")
else:
    for col, count in nan_counts.head(10).items():
        print(f"    {col}: {count}")

In [ ]:
# Save combined dataset
output_path = os.path.join(OUTPUT_DIR, "ctrate_train_combined.csv")
df_combined.to_csv(output_path, index=False)

print(f"\n✓ Dataset saved: {output_path}")
print(f"✓ WP1 complete")

In [ ]:
# Generate markdown documentation
report = []
report.append("# WP1: CT-RATE Data Ingestion - Findings\n\n")
report.append(f"## Dataset Overview\n")
report.append(f"- Total reports: {len(df_combined)}\n")
report.append(f"- Total columns: {len(df_combined.columns)}\n")
report.append(f"- Columns: {', '.join(df_combined.columns.tolist()[:20])}...\n\n")

report.append(f"## Sample Report Analysis (index {sample_idx})\n")
report.append(f"**VolumeName:** `{sample['VolumeName']}`\n\n")

for field in ['ClinicalInformation_EN', 'Technique_EN', 'Findings_EN', 'Impressions_EN']:
    text = str(sample[field])
    word_count = len(text.split())
    char_count = len(text)
    report.append(f"### {field}\n")
    report.append(f"**Stats:** {word_count} words, {char_count} characters\n\n")
    report.append(f"```\n{textwrap.fill(text, width=80)}\n```\n\n")

report.append("### Metadata\n")
metadata_fields = ['StudyDate', 'PatientSex', 'PatientAge', 'Manufacturer', 
                   'SeriesDescription', 'CTDIvol', 'NumberofSlices']
for field in metadata_fields:
    report.append(f"- **{field}:** {sample[field]}\n")
report.append("\n")

report.append("## Structural Checks\n")
report.append(f"- Duplicate VolumeNames: {df_combined['VolumeName'].duplicated().sum()}\n\n")

report.append("### Missing Values\n")
if nan_counts.empty:
    report.append("- No missing values found\n\n")
else:
    for col, count in nan_counts.items():
        report.append(f"- `{col}`: {count} missing\n")
    report.append("\n")

report.append("## Key Fields for Clinical Query\n")
relevant_fields = {
    "Patient": ["PatientSex", "PatientAge"],
    "Report Text": ["ClinicalInformation_EN", "Technique_EN", "Findings_EN", "Impressions_EN"],
    "Study Metadata": ["StudyDate", "Manufacturer", "SeriesDescription", "CTDIvol"],
    "Pathology Labels": list(df_labels.columns[1:])[:5] + ["..."]
}
for category, fields in relevant_fields.items():
    report.append(f"- **{category}:** {', '.join(fields)}\n")

with open("wp1_findings.md", "w", encoding="utf-8") as f:
    f.writelines(report)

print("✓ wp1_findings.md saved")